# Apollo Lead Importer
Loads Apollo CSV exports → scrapes websites → summarises → embeds → saves to `all-contacts/apollo/`.

## 1. Environment

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q playwright openai anthropic tiktoken tldextract duckdb aiohttp
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.2 MB/s eta 0:00:00
(node:613) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 41.4s167.3 MiB [] 0% 14.1s167.3 MiB [] 0% 12.5s167.3 MiB [] 0% 6.9s167.3 MiB [] 1% 4.2s167.3 MiB [] 2% 4.0s167.3 MiB [] 3% 3.7s167.3 MiB [] 4% 3.2s167.3 MiB [] 5% 2.8s167.3 MiB [] 5% 2.6s167.3 MiB [] 6% 2.6s167.3 MiB [] 7% 2.5s167.3 MiB [] 8% 2.4s167.3 MiB [] 9% 2.2s167.3 MiB [] 9% 2.3s167.3 MiB [] 10% 2.2s167.3 MiB [] 12% 2.1s167.3 MiB [] 12% 2.2s167.3 MiB [] 13% 2.1s167.3 MiB [] 14% 1.9s167.3 MiB [] 14% 2.0s167.3 MiB []

In [10]:
!apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libcups2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2 is already the newest version (1.2.6.1-1ubuntu1).
libasound2 set to manually installed.
libcairo2 is already the newest version (1.16.0-5ubuntu2).
libcairo2 set to manually installed.
libxdamage1 is already the newest version (1:1.1.5-2build2).
libxdamage1 set to manually installed.
libxfixes3 is already the newest version (1:6.0.0-1).
libxfixes3 set to manually installed.
libxkbcommon0 is already the newest version (1.4.0-1).
libxkbcommon0 set to manually installed.
libxrandr2 is already the newest version (2:1.5.2-1build1).
libxrandr2 set to manually installed.
libcups2 is already the newest version (2.4.1op1-1ubuntu4.16).
libcups2 set to manually installed.
libgbm1 is already the newest version (23.2.1-1ubuntu3.1~22.04.3).
libgbm1 set to manually installed.
libpango-1.0-0 is already the newest version (1.50.6+ds-2ubuntu1).
libpango-1.0-0 set to manually installed.
The followin

In [3]:
%cd '/content/drive/Shareddrives/Matcher/notebooks/pipelines/leads-import'

/content/drive/Shareddrives/Matcher/notebooks/pipelines/leads-import


## 2. Imports

In [4]:
import sys
import uuid
import pandas as pd
from datetime import datetime
from openai import AsyncOpenAI

sys.path.append('/content/drive/Shareddrives/EXT Matcher')
from src.modules.Embedding.text_embedder import TextProcessor
from src.modules.lead_importer import (
    load_csvs,
    normalize_columns,
    filter_non_commercial,
    dedup_against_existing,
    scrape_pages,
    summarize_companies,
    embed_summaries,
    export_contacts,
)

## 3. Config

In [5]:
KEYS_PATH   = '/content/drive/Shareddrives/Matcher/keys'
INPUT_GLOB  = 'data/unprocessed/apollo/*'
YEAR_FILTER = None   # only load files containing this string; set None to load all
EXPORT_PATH = '/content/drive/Shareddrives/Matcher/notebooks/pipelines/data/all-contacts/apollo'
EXISTING_CONTACTS_GLOB = '/content/drive/Shareddrives/Matcher/notebooks/pipelines/data/all-contacts/**/*.parquet'

today = datetime.today().strftime('%Y-%m-%d')

with open(f'{KEYS_PATH}/openai_key.txt') as f:
    _OAI_KEY = f.read().strip()

openai_client = AsyncOpenAI(api_key=_OAI_KEY)
tp = TextProcessor(f'{KEYS_PATH}/openai_key.txt')

## 4. Load & normalise

In [6]:
raw = load_csvs(INPUT_GLOB, year_filter=YEAR_FILTER)
raw = normalize_columns(raw)
print(f'Loaded: {len(raw)} rows')
print('Columns:', raw.columns.sort_values().tolist())

Loaded: 4000 rows
Columns: ['_employees', 'account_owner', 'account_stage', 'annual_revenue', 'apollo_account_id', 'company_address', 'company_city', 'company_country', 'company_linkedin_url', 'company_name', 'company_name_for_emails', 'company_phone', 'company_postal_code', 'company_state', 'company_street', 'facebook_url', 'founded_year', 'industry', 'keywords', 'last_raised_at', 'latest_funding', 'latest_funding_amount', 'lists', 'logo_url', 'naics_codes', 'number_of_retail_locations', 'short_description', 'sic_codes', 'subsidiary_of', 'technologies', 'total_funding', 'twitter_url', 'website']


## 5. Standardise schema

In [7]:
APOLLO_RENAMES = {
    'company':             'company_name',
    'industry':            'segment',
    'website':             'companyWebsite',
    'company_linkedin_url': 'linkedin',
}
KEEP_COLS = ['company_name', 'company_name_for_emails', 'segment',
             'companyWebsite', 'linkedin', 'short_description']

df = (
    raw
    .rename(columns=APOLLO_RENAMES)
    .assign(company_name=lambda x: x['company_name'].astype(str))
    .sort_values('company_name', ascending=False)  # puts LLC / Inc first for dedup
    .drop_duplicates(subset='companyWebsite', keep='first')
    .reset_index(drop=True)
    [[c for c in KEEP_COLS if c in raw.rename(columns=APOLLO_RENAMES).columns]]
)

# separate contacts with no website — save as runoff at the end
runoff = df[df['companyWebsite'].isna()].reset_index(drop=True)
df     = df[df['companyWebsite'].notna()].reset_index(drop=True)

df = filter_non_commercial(df)
print(f'After schema normalisation: {len(df)} rows | Runoff (no website): {len(runoff)}')

After schema normalisation: 3301 rows | Runoff (no website): 1


## 6. Dedup against existing contacts

In [8]:
df = dedup_against_existing(df, EXISTING_CONTACTS_GLOB)
print(f'After dedup: {len(df)} new contacts')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dedup: 3301 → 3150 rows (151 already in contacts)
After dedup: 3150 new contacts


## 7. Scrape pages

In [11]:
page_texts = await scrape_pages(df)

Scraping 3150 URLs...

Stage 1: aiohttp
  ✓ aiohttp http://www.renu.earth
  ✓ aiohttp http://www.kmvtech.com
  ✓ aiohttp http://www.ia-robotics.com
  ✓ aiohttp http://www.studio3e8.com
  ✓ aiohttp http://www.iengineeredsolutions.com
  ✓ aiohttp http://www.brain4.care
  ✓ aiohttp http://www.pdblowers.com
  ✓ aiohttp http://www.strato-cloud.io
  ✓ aiohttp http://www.sys-tek.com
  ✓ aiohttp http://www.blu-power.com
  ✓ aiohttp http://www.e-motion-hybrid.com
  ✓ aiohttp http://www.escoadvisor.com
  ✓ aiohttp http://www.vilogics.com
  ✓ aiohttp http://www.pinta-acoustic.com
  ✓ aiohttp http://www.zerov.io
  ✓ aiohttp http://www.zambahvac.com
  ✓ aiohttp http://www.zemanmfg.com
  ✓ aiohttp http://www.yosemiteclean.com
  ✓ aiohttp http://www.zodiac.com
  ✓ aiohttp http://www.dais.co
  ✓ aiohttp http://www.ziegleres.com
  ✓ aiohttp http://www.zero-max.com
  ✓ aiohttp http://www.zamma.com
  ✓ aiohttp http://www.zanesvillesteel.com
  ✓ aiohttp http://www.zippertubing.com
  ✓ aiohttp http://www.y

### 7a. Optional: re-scrape failures
Rerun this cell if you want to retry any remaining FAILEDs before moving on.

In [12]:
failed_urls = page_texts[page_texts['page_text'] == 'FAILED']['companyWebsite'].tolist()
print(f'{len(failed_urls)} still failed')
retry = await scrape_pages(pd.DataFrame({'companyWebsite': failed_urls}))
page_texts.update(retry.set_index('companyWebsite'), overwrite=True)

0 still failed
Scraping 0 URLs...
  No URLs to scrape — check that companyWebsite column is populated.


## 8. Merge page text + short_description

In [16]:
df_merged = pd.merge(df, page_texts, on='companyWebsite', how='left')

# combine Apollo's short_description with scraped page text
df_merged['page_text'] = (
    df_merged['short_description'].fillna('').astype(str) + '\n' +
    df_merged['page_text'].fillna('')
)

# add rows that failed scraping AND have no description to runoff
bad_mask = df_merged['short_description'].isna() & df_merged['page_text'].str.contains('ERROR', na=False)
runoff   = pd.concat([runoff, df_merged[bad_mask]]).reset_index(drop=True)
df_merged = df_merged[~bad_mask].reset_index(drop=True)

print(f'Ready for summarisation: {len(df_merged)} | Runoff: {len(runoff)}')

Ready for summarisation: 3143 | Runoff: 15


## 9. Summarise with GPT-4o-mini

In [17]:
df_merged = await summarize_companies(df_merged, openai_client)

  ✓ http://www.ensol.energy                                 | 3142 remaining
  ✓ http://www.agrobotics.com                               | 3141 remaining
  ✓ http://www.studio3e8.com                                | 3140 remaining
  ✓ http://www.zimmererkubota.com                           | 3139 remaining
  ✓ http://www.schober.engineer                             | 3138 remaining
  ✓ http://www.eufymake.com                                 | 3137 remaining
  ✓ http://www.blu-power.com                                | 3136 remaining
  ✓ http://www.dflow.sh                                     | 3135 remaining
  ✓ http://www.kmvtech.com                                  | 3134 remaining
  ✓ http://www.zerocows.com                                 | 3133 remaining
  ✓ http://www.hypoflo.com                                  | 3132 remaining
  ✓ http://www.iba-ag.com                                   | 3131 remaining
  ✓ http://www.www.org.cn                                   | 3130 remaining

## 10. Generate embeddings

In [18]:
df_merged = await embed_summaries(df_merged, tp)

  ✓ embedded http://www.vilogics.com
  ✓ embedded http://www.tikimundo.com
  ✓ embedded http://www.studio3e8.com
  ✓ embedded http://www.sys-tek.com
  ✓ embedded http://www.www.org.cn
  ✓ embedded http://www.schober.engineer
  ✓ embedded http://www.strato-cloud.io
  ✓ embedded http://www.pinta-acoustic.com
  ✓ embedded http://www.pdblowers.com
  ✓ embedded http://www.kmvtech.com
  ✓ embedded http://www.renu.earth
  ✓ embedded http://www.mbs-ac.de
  ✓ embedded http://www.iba-ag.com
  ✓ embedded http://www.in2being.com
  ✓ embedded http://www.avori.energy
  ✓ embedded http://www.iengineeredsolutions.com
  ✓ embedded http://www.ia-robotics.com
  ✓ embedded http://www.punkerusa.com
  ✓ embedded http://www.eufymake.com
  ✓ embedded http://www.hypoflo.com
  ✓ embedded http://www.esilo.com
  ✓ embedded http://www.escoadvisor.com
  ✓ embedded http://www.dassoxtr.com
  ✓ embedded http://www.demaximis.com
  ✓ embedded http://www.dflow.sh
  ✓ embedded http://www.dais.co
  ✓ embedded http://www.e-

## 11. Export

In [19]:
export_contacts(df_merged, EXPORT_PATH, source_name='apollo', today=today)

Exported 264 rows → /content/drive/Shareddrives/Matcher/notebooks/pipelines/data/all-contacts/apollo/apollo_093d3_2026-02-26.parquet


'/content/drive/Shareddrives/Matcher/notebooks/pipelines/data/all-contacts/apollo/apollo_093d3_2026-02-26.parquet'

In [21]:
# Save runoff
if len(runoff) > 0:
    runoff_path = f'/content/drive/Shareddrives/Matcher/notebooks/pipelines/leads-import/data/unprocessed/apollo/runoff_{uuid.uuid4().hex[:5]}_{today}.xlsx'
    runoff.to_excel(runoff_path, index=False)
    print(f'Runoff saved → {runoff_path}  ({len(runoff)} rows)')
else:
    print('No runoff.')

Runoff saved → /content/drive/Shareddrives/Matcher/notebooks/pipelines/leads-import/data/unprocessed/apollo/runoff_7981b_2026-02-26.xlsx  (15 rows)
